# Open Notebook & Additional Resources

<a target="_blank" href="https://colab.research.google.com/github/Nicolepcx/hands-on-multimodal-AI/blob/main/hands-on/DEMO_session_02_video_classification_example.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>
<a target="_blank" href="https://www.oreilly.com/library/view/transformers-the-definitive/9781098167004/">
  <img src="https://img.shields.io/badge/AI%20Agents%20Book-Read%20on%20O'Reilly-d40101?style=flat" alt="Transformers: The Definitive Guide"/>
</a>




# About this Notebook

Multimodal Video Understanding with Qwen3-VL



This notebook demonstrates how to:

1. Download a video from Google Drive
2. Render the video inline in a Jupyter/Colab notebook
3. Load a multimodal vision-language model
4. Perform video-based instruction following
5. Generate a structured textual description from video input


## Why do you use `av`?

The Qwen3-VL model internally decodes video frames. The `av` library enables efficient video processing and frame extraction.

This is critical for:

* Frame sampling
* Temporal understanding
* Efficient video tokenization


You convert the MP4 file to a base64 data URL and embed it inside an HTML `<video>` tag.

This is useful for:

* Debugging visual inputs
* Verifying model input fidelity

In production systems, you would not embed base64 like this. You would pass a file path or stream handle.


Key parameters:

### `max_pixels`

Controls spatial resolution per frame.

Lower → faster inference
Higher → better visual detail

### `max_frames`

Controls temporal sampling.

Lower → faster
Higher → better motion understanding

This is a direct example of **compute-quality tradeoff** in multimodal systems.


# Production Considerations

If you were building this into a real system:

• Avoid base64 rendering
• Stream frames instead of full video files
• Dynamically adjust `max_frames` based on video length
• Add safety filters
• Add structured output (JSON mode)
• Log frame selection for observability


In [ ]:
!pip install av

In [ ]:
import base64
from IPython.display import HTML
from IPython.display import Video
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor
import torch
import gdown
import os

In [ ]:
file_id = "1g2a2-HLEuwWFMsoDD1iqTx2NBuiCDSDD"
destination = "video_from_drive.mp4"
!gdown --id {file_id} -O {destination}

# Check if the file was downloaded
import os
if os.path.exists(destination):
    print(f"File downloaded successfully: {destination}")
else:
    print("Download failed.")


/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:140: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From: https://drive.google.com/uc?id=1g2a2-HLEuwWFMsoDD1iqTx2NBuiCDSDD
To: /content/video_from_drive.mp4
100% 2.43M/2.43M [00:00<00:00, 105MB/s]
File downloaded successfully: video_from_drive.mp4


In [ ]:
with open('/content/video_from_drive.mp4','rb') as f:
    mp4 = f.read()
data_url = "data:video/mp4;base64," + base64.b64encode(mp4).decode()
HTML(f'<video controls width="640" height="360" src="{data_url}"></video>')

In [ ]:

drive_url = "https://drive.google.com/uc?id=1g2a2-HLEuwWFMsoDD1iqTx2NBuiCDSDD"
video_path = "video_from_drive.mp4"
gdown.download(drive_url, video_path, quiet=False)

assert os.path.exists(video_path), "Video download failed"


model = Qwen3VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen3-VL-8B-Instruct",
    dtype="auto",
    device_map="auto",
)

processor = AutoProcessor.from_pretrained("Qwen/Qwen3-VL-8B-Instruct")


messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "video",
                "video": video_path,
                # optional limits this can be omitted or tuned
                "max_pixels": 128 * 32 * 32,
                "max_frames": 16,
            },
            {
                "type": "text",
                "text": "Briefly classify and describe what happens in this video.",
            },
        ],
    }
]


inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt",
)

inputs = inputs.to(model.device)


with torch.no_grad():
    generated_ids = model.generate(**inputs, max_new_tokens=128)

generated_ids_trimmed = [
    out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
]

output_text = processor.batch_decode(
    generated_ids_trimmed,
    skip_special_tokens=True,
    clean_up_tokenization_spaces=False,
)

print(output_text[0])


Downloading...
From: https://drive.google.com/uc?id=1g2a2-HLEuwWFMsoDD1iqTx2NBuiCDSDD
To: /content/video_from_drive.mp4
100%|██████████| 2.43M/2.43M [00:00<00:00, 90.0MB/s]


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/transformers/video_processing_utils.py:879: UserWarning: `torchcodec` is not installed and cannot be used to decode the video by default. Falling back to `torchvision`. Note that `torchvision` decoding is deprecated and will be removed in future versions. 
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/video_utils.py:524: UserWarning: Using `torchvision` for video decoding is deprecated and will be removed in future versions. Please use `torchcodec` instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/io/_video_deprecation_warning.py:9: UserWarning: The video decoding and encoding capabilities of torchvision are deprecated from version 0.22 and will be removed in version 0.24. We recommend that you migrate to TorchCodec, where we'll consolidate the future decoding/encoding capabilities of PyTorch: https://github.com/pytorch/torchcodec
  warnings.warn(


Two men are playing basketball on an outdoor court. The man in the red jersey dribbles the ball while the man in the black jersey defends him. The man in red makes a move to the left, causing the defender to shift his position. The camera follows the action closely, capturing the intensity of the game.
